In [ ]:
import pandas as pd

In [ ]:
# Load the cleaned transaction data exported from PostgreSQL
df = pd.read_csv('/content/RetailCleaned.csv', low_memory=False)

# Convert invoice dates for time-based RFM calculations
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

# Verify dataset dimensions, column types, and missing values
print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1033036 non-null  object        
 1   stockcode    1033036 non-null  object        
 2   description  1028761 non-null  object        
 3   quantity     1033036 non-null  int64         
 4   invoicedate  1033036 non-null  datetime64[ns]
 5   price        1033036 non-null  float64       
 6   customer_id  797885 non-null   float64       
 7   country      1033036 non-null  object        
 8   sourcesheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 70.9+ MB


In [ ]:
# Keep only identified customers with positive purchase quantities
rfm_data = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()

# Use the day after the last transaction as the analysis date
snapshot_date = pd.to_datetime('2011-12-10')

# Calculate RFM metrics for each customer
rfm = rfm_data.groupby('customer_id').agg(
    Recency=('invoicedate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('invoice', 'nunique'),
    Monetary=('quantity', lambda x: (x * rfm_data.loc[x.index, 'price']).sum())
).reset_index()

# Verify the customer-level RFM table
print(rfm.shape)
rfm.head()

(5881, 4)


,customer_id,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [ ]:
# Score recency: more recent purchases receive higher scores
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Score frequency: rank first to handle repeated purchase counts
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Score monetary value: higher spending receives higher scores
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Assign business segments based on RFM scores
def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Customers inactive for over 90 days are flagged as churned
rfm['IsChurned'] = rfm['Recency'] > 90

# Verify the scores, segments, and churn classification
print(rfm.shape)
rfm.head()

(5881, 9)


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,At Risk,True
1,12347.0,2,8,4921.53,5,4,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,Loyal,False
3,12349.0,18,4,4428.69,5,3,5,Loyal,False
4,12350.0,310,1,334.40,2,1,2,Lost,True


In [ ]:
# Keep valid customer purchases for Power BI analysis
df = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()

# Create transaction month for monthly sales analysis
df['InvoiceMonth'] = df['invoicedate'].dt.to_period('M').dt.to_timestamp()

# Calculate revenue for each transaction
df['Revenue'] = df['quantity'] * df['price']

# Attach customer-level RFM scores, segments, and churn status
df_final = df.merge(
    rfm[['customer_id', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='customer_id',
    how='left'
)

# Verify the final dataset before exporting
print(df_final.shape)
df_final.head()

(779495, 19)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,sourcesheet,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011,2009-12-01,59.5,157,8,2433.28,3,4,4,Loyal,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
3,489434,22041,RECORD FRAME 7 SINGLE SIZE,48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011,2009-12-01,100.8,157,8,2433.28,3,4,4,Loyal,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011,2009-12-01,39.6,157,8,2433.28,3,4,4,Loyal,True


In [ ]:
# Export the final transaction and RFM dataset for Power BI
df_final.to_excel(
    '/content/EcommerceDashboardData.xlsx',
    index=False
)

# Confirm successful export
print("Excel export completed successfully!")

Excel export completed successfully!
